<a href="https://colab.research.google.com/github/cleisonlima/Atividades/blob/main/Integra%C3%A7%C3%A3o_com_a_API_do_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Definir Idioma

Certifique-se de que a variável 'language' esteja definida como 'pt' para especificar o idioma de transcrição e geração de texto.

In [ ]:
language = 'pt'

## Gravar Áudio

Execute a célula para gravar seu áudio. O sistema aguardará por 5 segundos de fala e salvará o arquivo como 'request_audio.wav'. Fale no microfone por 5 segundos. O áudio gravado será salvo como 'request_audio.wav' e reproduzido automaticamente após a gravação.

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});

var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  await sleep(time);

  recorder.onstop = async () => {
    const blob = new Blob(chunks);
    const text = await b2text(blob);
    resolve(text);
  };

  recorder.stop();
});
"""

def record(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js(f'record({sec * 1000})')
    audio = b64decode(js_result.split(',')[1])

    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)

    return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))

Ouvindo...



<IPython.core.display.Javascript object>

## Instalar Whisper

Instale a biblioteca Whisper do OpenAI. Este passo é crucial para a funcionalidade de Speech-to-Text.

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Transcrever Áudio com Whisper

Utilize o modelo Whisper para converter o áudio gravado em texto. O resultado da transcrição será armazenado na variável 'result'.

In [ ]:
import whisper

model = whisper.load_model("small")

# Opção 1: definir explicitamente o idioma
language = "pt"

result = model.transcribe(
    record_file,
    fp16=False,
    language=language
)

print(result["text"])

## Instalar OpenAI

Garanta que a biblioteca OpenAI esteja instalada para interagir com a API do ChatGPT.

In [ ]:
!pip install openai

## Obter Resposta do ChatGPT

Envie o texto transcrito pelo Whisper para a API do ChatGPT e obtenha uma resposta. É importante resolver qualquer 'RateLimitError' externo na sua conta OpenAI para que este passo funcione.

In [ ]:
import openai

# Initialize the OpenAI client with the API key
client = openai.OpenAI(api_key="")

# Certifique-se de que a célula que define 'result' (e.g., a célula de transcrição do Whisper, cell_id: 1Lua2dsKUDsR) foi executada.
try:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": result["text"]}]
    )
    chat_response = response.choices[0].message.content
    print(chat_response)
except openai.RateLimitError:
    print("Error: Unable to obtain chat_response due to OpenAI API RateLimitError.\nPlease check your OpenAI account's usage and billing details at https://platform.openai.com/account/usage to resolve the quota issue.")
    chat_response = None
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    chat_response = None

Error: Unable to obtain chat_response due to OpenAI API RateLimitError.
Please check your OpenAI account's usage and billing details at https://platform.openai.com/account/usage to resolve the quota issue.


## Instalar gTTS

Instale a biblioteca Google Text-to-Speech (gTTS) que será usada para converter a resposta do ChatGPT em áudio.

In [ ]:
!pip install gTTS

## Gerar e Tocar Resposta em Áudio

Use gTTS para gerar um arquivo de áudio a partir da resposta do ChatGPT e, em seguida, toque este áudio para completar a experiência de conversação por voz.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

# Check if chat_response is available from previous steps
if 'chat_response' in locals() and chat_response:
    print(f"Generating audio for: {chat_response}")
    # Create a gTTS object
    tts = gTTS(text=chat_response, lang=language, slow=False)

    # Save the audio to a file
    audio_file_path = 'chat_response.mp3'
    tts.save(audio_file_path)
    print(f"Audio saved to {audio_file_path}")

    # Play the audio
    display(Audio(audio_file_path, autoplay=True))
else:
    print("Error: 'chat_response' is not defined. Please resolve the OpenAI API RateLimitError and regenerate the chat response before proceeding.")
    print("You can check your OpenAI account's usage and billing details at https://platform.openai.com/account/usage")

Error: 'chat_response' is not defined. Please resolve the OpenAI API RateLimitError and regenerate the chat response before proceeding.
You can check your OpenAI account's usage and billing details at https://platform.openai.com/account/usage
